In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from catboost import CatBoostClassifier, cv, Pool

### CatBoost: Categorical Boosting

**CatBoost** (Categorical Boosting) is developed by Yandex and has a huge advantage: it **handles categorical variables natively** without manual encoding!

#### CatBoost vs XGBoost vs LightGBM

| Feature | CatBoost | XGBoost | LightGBM |
|---------|----------|---------|----------|
| **Categorical Support** | ✅ Native | ❌ Manual | ❌ Manual |
| **Windows Stability** | ✅ Excellent | ✅ Excellent | ⚠️ Can be tricky |
| **Speed** | ✅ Fast | ✅ Fast | ⚡ Fastest |
| **Overfitting Prevention** | ✅ Built-in | Need tuning | Need tuning |
| **Ease of Use** | ✅ Very Easy | Moderate | Moderate |


In [3]:
data = pd.read_csv('../datasets/titanic.csv')
data.columns = [col.lower() for col in data.columns]
data

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [7]:
# data
data['title'] = data['name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())

# Group rare titles
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Dr': 'Rare',
    'Rev': 'Rare',
    'Col': 'Rare',
    'Major': 'Rare',
    'Mlle': 'Miss',
    'Mme': 'Mrs',
    'Ms': 'Miss',
    'Lady': 'Rare',
    'Sir': 'Rare',
    'Countess': 'Rare',
    'Jonkheer': 'Rare',
    'Don': 'Rare'
}
data['title'] = data['title'].map(title_mapping)
data['family_size'] = data['sibsp'] + data['parch'] + 1
data['is_alone'] =  (data['family_size'] == 1).astype(int)
data['cabin_prefix'] = data['cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'Unknown') # cabin prefix
data['age_group'] = pd.cut(data['age'], bins=[0, 12, 18, 30, 50, 100], labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Elder'])
data['fare_group'] = pd.cut(data['fare'], bins=[0, 10, 25, 50, 100, 600],labels=['Low', 'Medium', 'High', 'Very High', 'Luxury'])

features = [
    'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
    'embarked', 'title', 'family_size', 'is_alone'
]

categorical_features = ['sex', 'embarked', 'title']

X = data[features]
y = data['survived']

# missing values (CatBoost handles this too but let's do it anyway)
X_prepared = X.copy()
X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
X_prepared['fare'].fillna(X_prepared['fare'].median(), inplace=True)

# IMPORTANT: fill categorical NaN values with 'Unknown' (as string!)
X_prepared['embarked'].fillna('Unknown', inplace=True)
X_prepared['sex'].fillna('Unknown', inplace=True)
X_prepared['title'].fillna('Unknown', inplace=True)

# ensure categorical features are strings (not float64)
X_prepared['embarked'] = X_prepared['embarked'].astype(str)
X_prepared['sex'] = X_prepared['sex'].astype(str)
X_prepared['title'] = X_prepared['title'].astype(str)

# data split
X_train, X_test, y_train, y_test = train_test_split(X_prepared, y, test_size=0.2, random_state=42, stratify=y)

# model (CatBoost handles categoricals natively!)
model = CatBoostClassifier(
    iterations=100,          # Number of trees
    learning_rate=0.1,
    depth=5,                 # Tree depth (similar to max_depth)
    cat_features=categorical_features,  # ← CRITICAL: Specify categoricals!
    verbose=0,               # Suppress output
    random_seed=42
)
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
y_pred_proba = model.predict_proba(X_test)

C:\Users\rah\AppData\Local\Temp\ipykernel_12940\3916150037.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
C:\Users\rah\AppData\Local\Temp\ipykernel_12940\3916150037.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

In [22]:
target_names = ['not survived', 'survived']

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


print("\n" + "="*60)
print("CATBOOST BUILT-IN CROSS-VALIDATION")
print("="*60)

# CatBoost's native CV instead of sklearn cv (doesn't have the cloning issue)
cv_params = {
    'iterations': 100,
    'learning_rate': 0.1,
    'depth': 5,
    'verbose': False,
    'random_seed': 42,
    'loss_function': 'Logloss',
}

train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=categorical_features
)

cv_results = cv(
    params=cv_params,
    pool=train_pool,
    fold_count=5,
    verbose=False
)

test_loss_mean = cv_results['test-Logloss-mean']
test_loss_std = cv_results['test-Logloss-std']
train_loss_mean = cv_results['train-Logloss-mean']
train_loss_std = cv_results['train-Logloss-std']

print(f"\nCV Results (Logloss - Lower is Better):")
print(f"Test Logloss Mean: {test_loss_mean.iloc[-1]:.4f}")
print(f"Test Logloss Std: {test_loss_std.iloc[-1]:.4f}")
print(f"Train Logloss Mean: {train_loss_mean.iloc[-1]:.4f}")
print(f"Train Logloss Std: {train_loss_std.iloc[-1]:.4f}")

# Logloss doesn't directly convert to accuracy, but we can estimate
# Lower logloss generally means better accuracy
print(f"\nCV Scores (Logloss - Lower is Better):")
print(f"CV Scores: {test_loss_mean.values}")

# Find best iteration (lowest test logloss)
best_iteration = test_loss_mean.idxmin()
print(f"Best iteration: {best_iteration}")
print(f"Best Test Logloss: {test_loss_mean.min():.4f}")


MODEL EVALUATION
Training Accuracy: 84.69%
Test Accuracy: 81.01%
Overfitting Gap: 0.0369
⚠️  MODERATE OVERFITTING (3-5%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

not survived       0.81      0.90      0.85       110
    survived       0.81      0.67      0.73        69

    accuracy                           0.81       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.81      0.81      0.81       179


Confusion Matrix:
[[99 11]
 [23 46]]
Correct predictions: 145
Total predictions: 179
Accuracy: 0.8101
Accuracy: 81.01%

not survived Accuracy: 0.9000 (90.00%)
survived Accuracy: 0.6667 (66.67%)

CATBOOST BUILT-IN CROSS-VALIDATION
Training on fold [0/5]

bestTest = 0.4096723658
bestIteration = 57

Training on fold [1/5]

bestTest = 0.451549938
bestIteration = 53

Training on fold [2/5]

bestTest = 0.3506682552
bestIteration = 91

Training on fold [3/5]

bestTest = 0.4093185354
bestIteration = 39

In [23]:
feature_importance = pd.DataFrame({
    'feature': X_prepared.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
feature_importance

,feature,importance
1,sex,44.143223
0,pclass,14.578224
5,fare,10.503570
7,title,10.009077
2,age,9.495129
6,embarked,3.440776
8,family_size,3.186075
3,sibsp,2.500132
4,parch,1.826980
9,is_alone,0.316814
